In [ ]:
# ── Cell 1 : Setup & Imports ──────────
import json
import pickle
import pathlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

project_root = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "pyproject.toml").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src import marginal_comparison as mc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 4)
print("Imports successful!")

In [ ]:
# ── Cell 2 : Locate Sibling Sampler Runs ──────────
# This notebook sits in a <chains>/<k>_comp/ folder and compares the NUTS, HMC and
# bayesm runs that live beside it: <sampler>/<run>/results/posterior_raw.pkl.

def _resolve_dir():
    nb = globals().get("__vsc_ipynb_file__")
    return pathlib.Path(nb).resolve().parent if nb else pathlib.Path.cwd()

XCOMP_DIR = _resolve_dir()
hits = sorted(XCOMP_DIR.glob("*/*/results/posterior_raw.pkl"))
if not hits:
    raise FileNotFoundError(
        f"No <sampler>/<run>/results/posterior_raw.pkl under:\n  {XCOMP_DIR}\n"
        f"This notebook must sit in a <chains>/<k>_comp/ folder. In VS Code set\n"
        f'  "jupyter.notebookFileRoot": "${{fileDirname}}"  and restart the kernel.'
    )

runs = {}
for h in hits:
    sampler = h.parents[2].name          # NUTS / HMC / bayesm
    runs.setdefault(sampler, h.parent)   # first run's results dir per sampler

meta = json.load(open(next(iter(runs.values())) / "meta.json"))
SCENARIO = meta["scenario"]
K_MODEL  = int(meta["k_model"])
K_TRUE   = int(meta["k_true"])
CHAINS   = int(meta["chains"])

print(f"x_comp folder : {XCOMP_DIR}")
print(f"scenario={SCENARIO}  K_MODEL={K_MODEL}  K_TRUE={K_TRUE}")
print(f"samplers found: {list(runs)}")

models = [mc.load_sampler(rd, name) for name, rd in runs.items()]
for m in models:
    print(f"  {m['name']:<8} mu {m['mu'].shape}")

In [ ]:
# ── Cell 3 : Ground Truth & Grids (Full, unbounded  +  Chebyshev-filtered) ──────────
# Two grids are built per parameter and carried through every cell below. Neither
# uses the True DGP to set bounds beyond including it in the envelope/moments -
# it stays an overlay only.
#   "Full"                   - mc.build_grids_full: raw min/max envelope over EVERY
#                               component (all K, incl. surplus/empty ones) of every
#                               sampler and the True DGP. Nothing is excluded, so every
#                               metric integrates over the ENTIRE marginal - but a
#                               diffuse empty component (K_MODEL > K_TRUE) or a sampler
#                               exploring far off the high-density area can stretch this
#                               range enormously and squash the real mass into a few
#                               pixels.
#   "Chebyshev (k=5, >=96%)" - mc.build_grids_chebyshev: clipped to each model's own
#                               AGGREGATE mixture [mean - 5*std, mean + 5*std] (Eq.
#                               5.5.2). Chebyshev's inequality, P(|X-mean|>=k*std) <=
#                               1/k**2, holds for ANY distribution with finite variance
#                               (no normality/unimodality assumption - the invariant
#                               marginal is itself a mixture) - so >=1 - 1/5**2 = 96% of
#                               each model's own marginal mass is guaranteed to lie
#                               inside its window, trimming the outlier-driven tails of
#                               "Full" while keeping a distribution-free coverage bound.
raw = json.load(open(project_root / "data" / "simulated" / "mixture" / f"{SCENARIO}.json"))
P = int(raw["n_params"])
param_names = raw.get("param_names") or [f"Param_{i}" for i in range(P)]
true_model = mc.true_dgp_model(raw)

GRIDS = {
    "Full":                  mc.build_grids_full(models, true_model, n_grid=1000, n_sigma=6),
    "Chebyshev (k=5, >=96%)": mc.build_grids_chebyshev(models, true_model, n_grid=1000, k=5.0),
}
for label, grids in GRIDS.items():
    print(f"Grid extents [{label}]:")
    for j, pj in enumerate(param_names):
        print(f"  {pj:<10} [{grids[j][0]:+.2f}, {grids[j][-1]:+.2f}]")

In [ ]:
# ── Cell 4 : Marginal Densities (Rossi Eq. 5.5.19) - Full vs Chebyshev-filtered ──────────
# True DGP = black (dashed reference), bayesm = red, Liesel samplers = distinct
# shades of blue (ColorBrewer Blues: dark -> light). One figure per grid pass.
# `marginal_density` is O(R*K*n_grid) per model - computed ONCE per (model, grid)
# pass here and cached in DENS/DENS_TRUE so Cells 5 and 7 reuse it instead of
# recomputing it internally.
colors = {"NUTS": "#08519c", "HMC": "#4292c6", "iwls": "#9ecae1", "bayesm": "#d62728"}
TRUE_COLOR = "#000000"
ncols = min(P, 2)
nrows = int(np.ceil(P / ncols))

DENS, DENS_TRUE = {}, {}
for label, grids in GRIDS.items():
    dens      = {m["name"]: mc.marginal_density(m, grids) for m in models}
    dens_true = mc.marginal_density(true_model, grids)
    DENS[label], DENS_TRUE[label] = dens, dens_true

    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6.2, nrows * 3.4), squeeze=False)
    ax = axes.ravel()
    for j, pj in enumerate(param_names):
        for name, d in dens.items():
            ax[j].plot(grids[j], d[j], lw=1.8, label=name, color=colors.get(name, "#7f7f7f"))
        ax[j].plot(grids[j], dens_true[j], lw=2.0, ls="--", color=TRUE_COLOR, label="True DGP")
        ax[j].set_title(pj, fontweight="bold")
        ax[j].set_yticks([])
    for j in range(P, len(ax)):
        ax[j].set_visible(False)
    ax[0].legend(fontsize=9)
    fig.suptitle(f"Marginal posterior densities of beta  (Rossi Eq. 5.5.19)  \u2014  {label}",
                 fontweight="bold")
    fig.tight_layout()
    plt.show()

In [ ]:
# ── Cell 5 : Density Distances vs True DGP - Full vs Chebyshev-filtered ──────────
# Every sampler's marginal is compared ONLY to the True DGP marginal (never to
# another sampler). Metrics: Hellinger (primary), KL(model||true), Jensen-Shannon,
# total-variation, Wasserstein-1. All label-invariant - relabeling/ECR would not
# change them. The two passes differ only in the integration grid built in Cell 3;
# distances typically shrink under "Chebyshev" once the diffuse/outlier tails that
# dominate "Full" are clipped out. Reuses the densities cached in Cell 4 (DENS,
# DENS_TRUE) rather than recomputing marginal_density a second time.
for label, grids in GRIDS.items():
    print(f"--- {label} ---")
    display(mc.distance_table(models, true_model, grids, param_names,
                               dens=DENS[label], dens_true=DENS_TRUE[label]))

In [ ]:
# ── Cell 6 : Mixture Moments (Rossi Eq. 5.5.2) & Weights ──────────
# Grid-independent - computed analytically from the draws, not by integrating over
# either grid - so this runs once for both passes above.
allm = models + [true_model]
mean_tbl = pd.DataFrame({m["name"]: mc.mixture_moments(m)[0] for m in allm}, index=param_names)
var_tbl  = pd.DataFrame({m["name"]: np.diag(mc.mixture_moments(m)[1]) for m in allm}, index=param_names)
print("E[theta]  (overall mixture mean):")
display(mean_tbl.round(3))
print("diag Var[theta]  (overall mixture variance):")
display(var_tbl.round(3))

In [ ]:
# ── Cell 7 : Invariant Convergence - Marginal Density Series - Full vs Chebyshev-filtered ──────────
# ESS / R-hat (arviz rank-normalised split-R-hat) of the label-invariant per-draw
# marginal density, over the high-density region of each parameter. The density mask
# (density_threshold) is relative to each grid's own peak, so both passes restrict to
# their own high-density region even though "Full" spans a much wider range. Reuses
# the densities cached in Cell 4 (DENS) rather than recomputing marginal_density a
# third time just to rebuild the mask.
if CHAINS == 1:
    print("NOTE: single chain -> R-hat is SPLIT-R-hat (the one chain is halved); a")
    print("      WITHIN-chain check only (Vehtari et al. 2021; Stan; BDA3 sec. 11.4).")
    print("      It cannot detect multimodality a lone chain never explored - the")
    print("      between-chain R-hat comes from the 2-chain runs.\n")
for label, grids in GRIDS.items():
    print(f"=== Grid: {label} ===")
    for m in models:
        print(f"--- {m['name']} ---")
        display(mc.density_series_diagnostics(m, grids, param_names, n_eval=40,
                                               marg=DENS[label][m["name"]]).round(3))

In [ ]:
# ── Cell 8 : Invariant Convergence - Moment Series & Notes ──────────
# Grid-independent, like Cell 6 - runs once.
for m in models:
    print(f"--- {m['name']} ---")
    display(mc.moment_series_diagnostics(m, param_names).round(4))

print("\nNotes:")
print(" - Every quantity here (marginal density Eq. 5.5.19, moments Eq. 5.5.2) is")
print("   LABEL-INVARIANT: a per-draw permutation of components leaves it unchanged,")
print("   so relabeling/ECR is unnecessary and would give identical results.")
print(" - Two grids are compared throughout: 'Full' (unbounded envelope over every")
print("   component of every sampler + True DGP - nothing excluded, but can be very")
print("   wide when a sampler explores far off the high-density area or K_MODEL >")
print("   K_TRUE) and 'Chebyshev (k=5, >=96%)' (clipped to each model's own aggregate")
print("   mixture mean +/- 5 std - a distribution-free bound guaranteeing at least 96%")
print("   of that model's own marginal mass is retained, regardless of shape).")
print(" - R-hat/ESS use arviz rank-normalised split-R-hat on the real (chains,draws)")
print("   series (Vehtari et al. 2021). For 1-chain runs the single chain is split")
print("   into halves (Stan; BDA3 sec. 11.4) - a within-chain check only. bayesm's")
print("   seed-based chains are not over-dispersed, so its R-hat is a weaker test")
print("   than NUTS/HMC (per CLAUDE.md).")